
# SIH26139 — Phase 1 MVP: Hybrid QML vs. Classical on WDBC

**Goal of this notebook (per EXECUTION_PLAN.md, Phase 1):** get ONE honest, end-to-end
number for Control A (full hybrid quantum-classical model) vs. Control B (a
parameter-matched classical control), on a single dataset (WDBC), single seed.
No sweep, no multi-seed statistics yet — that's Phase 2. The point here is to
prove the pipeline works and produce a real, unfabricated result before any
UI or dashboard work starts.

**Design choice:** the hybrid model below is built with two toggles —
`entangle` and `quantum_trainable` — so the *exact same class* becomes
Control A, C, or D later (Phase 2) just by flipping flags. That's deliberate:
one codebase, one evaluation path, for all controls — which is what makes the
A/B/C/D comparison fair and defensible to a judge.

**Golden rule (from the blueprint review):** if this VQC-based hybrid model
is not training stably/reliably on WDBC after reasonable tuning, fall back to
the QSVM path (quantum kernel + classical SVM) implemented near the bottom of
this notebook — it has no gradient-training instability. Don't burn days
fighting a VQC that won't converge; the ablation study's scientific question
survives either way.

**Backend:** everything here runs on `default.qubit` / `lightning.qubit`
(PennyLane simulators) — no IBM hardware calls in this notebook. The real-QPU
run is a separate, later step (Phase 2/3) using these same trained weights
for inference only.


## 0. Setup

In [1]:

# Environment setup — self-healing, safe to re-run.
#
# Why this cell is written defensively: Kaggle's base image ships an OLD
# PennyLane together with a NEW `autoray`. Old PennyLane (<=0.39) does
# `class NumpyMimic(ar.autoray.NumpyMimic)` at import time, and that
# attribute was moved in autoray>=0.7, so `import pennylane` dies with
#   AttributeError: module 'autoray.autoray' has no attribute 'NumpyMimic'
# before any of our code runs.
#
# The fix is NOT to pin an old autoray to appease an old PennyLane — it's to
# install a CURRENT PennyLane, which is built against current autoray. We let
# pip resolve the whole set together so the versions are mutually consistent.
import sys, subprocess, importlib

subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "-U",
     "pennylane", "pennylane-qiskit", "qiskit-machine-learning"],
    check=True,
)

# Self-heal the running kernel. If an earlier run in THIS session already
# tried (and failed) to import the broken PennyLane, stale/partial module
# objects can linger in sys.modules and a fresh `import` would hand them
# back. Dropping them + invalidating the import caches makes the fix take
# effect immediately, with NO kernel restart required.
for _m in [m for m in list(sys.modules)
           if m.split(".")[0] in ("pennylane", "pennylane_qiskit", "autoray", "autograd")]:
    del sys.modules[_m]
importlib.invalidate_caches()

try:
    import pennylane as _pl
    print(f"Environment OK — PennyLane {_pl.version()}")
except Exception as _exc:  # last-resort fallback, should not trigger
    print(f"Import still failing ({type(_exc).__name__}: {_exc})")
    print("Restarting the kernel automatically. When it comes back: Run All again.")
    try:
        import IPython
        IPython.Application.instance().kernel.do_shutdown(True)
    except Exception:
        print("Auto-restart unavailable — use Kaggle menu: Run -> Restart & Run All")


Environment OK — PennyLane 0.45.1


In [2]:

import json
import time
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import pennylane as qml
from sklearn.datasets import load_breast_cancer  # this IS WDBC
from sklearn.decomposition import PCA
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.svm import SVC
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, confusion_matrix,
)

RESULTS_DIR = Path("results")
RESULTS_DIR.mkdir(exist_ok=True)

# ---- Reproducibility -------------------------------------------------------
SEED = 42

def set_seed(seed: int) -> None:
    np.random.seed(seed)
    torch.manual_seed(seed)

set_seed(SEED)

# ---- Fixed config for this MVP (Phase 1) -----------------------------------
N_QUBITS = 6      # qubit budget locked per EXECUTION_PLAN.md — keeps sim fast,
                  # keeps future real-hardware jobs short
N_LAYERS = 2      # shallow ansatz — deep circuits are unrealistic on NISQ hardware
DEVICE = "default.qubit"  # swap to "lightning.qubit" for the Phase 2 sweep (faster)

print(f"PennyLane {qml.version()}, torch {torch.__version__}")
print(f"Config: n_qubits={N_QUBITS}, n_layers={N_LAYERS}, seed={SEED}")


PennyLane 0.45.1, torch 2.14.0+cpu
Config: n_qubits=6, n_layers=2, seed=42


## 1. Data — WDBC (Wisconsin Diagnostic Breast Cancer)

`sklearn.datasets.load_breast_cancer` *is* WDBC (569 samples, 30 features,
benign/malignant) — no download needed, and it's the exact dataset used by
Gonaygunta (2026), your closest reference paper.


In [3]:

data = load_breast_cancer()
X_raw, y = data.data, data.target
# sklearn's target: 0 = malignant, 1 = benign — flip so 1 = malignant (the
# "positive"/high-risk class), which makes sensitivity/recall mean the right
# thing clinically (catching malignant cases).
y = 1 - y

print(f"WDBC: {X_raw.shape[0]} samples, {X_raw.shape[1]} raw features")
print(f"Class balance — malignant (positive): {y.mean():.3f}")

X_train_raw, X_test_raw, y_train, y_test = train_test_split(
    X_raw, y, test_size=0.2, stratify=y, random_state=SEED
)
print(f"Train: {X_train_raw.shape[0]}  Test: {X_test_raw.shape[0]}")


WDBC: 569 samples, 30 raw features
Class balance — malignant (positive): 0.373
Train: 455  Test: 114


In [4]:

# Preprocessing pipeline: standardize -> PCA down to the qubit budget
# -> scale to [0, pi] for angle encoding.
# IMPORTANT: fit scaler/PCA on TRAIN ONLY, transform test with the fitted
# objects — avoids test-set leakage.
scaler = StandardScaler().fit(X_train_raw)
X_train_std = scaler.transform(X_train_raw)
X_test_std = scaler.transform(X_test_raw)

pca = PCA(n_components=N_QUBITS, random_state=SEED).fit(X_train_std)
X_train_pca = pca.transform(X_train_std)
X_test_pca = pca.transform(X_test_std)
print(f"PCA explained variance ratio (sum): {pca.explained_variance_ratio_.sum():.3f}")

angle_scaler = MinMaxScaler(feature_range=(0, np.pi)).fit(X_train_pca)
X_train_enc = angle_scaler.transform(X_train_pca).astype(np.float32)
X_test_enc = angle_scaler.transform(X_test_pca).astype(np.float32)

X_train_t = torch.tensor(X_train_enc)
X_test_t = torch.tensor(X_test_enc)
y_train_t = torch.tensor(y_train, dtype=torch.float32)
y_test_t = torch.tensor(y_test, dtype=torch.float32)


PCA explained variance ratio (sum): 0.889


## 2. Quantum circuit — reusable across Controls A / C / D

One ansatz function, two toggles:
- `entangle=False` turns Control A into Control C (entanglement-ablated)
- quantum params frozen (`requires_grad=False` after construction) turns
  Control A into Control D (fixed/untrained quantum layer)

This is what makes the eventual ablation a fair, apples-to-apples comparison
instead of four different pieces of code.


In [5]:

def variational_layer(weights, wires, entangle: bool):
    """One layer: per-qubit RY+RZ rotation, optional ring of CNOTs."""
    for i, w in enumerate(wires):
        qml.RY(weights[i, 0], wires=w)
        qml.RZ(weights[i, 1], wires=w)
    if entangle and len(wires) > 1:
        for i in range(len(wires)):
            qml.CNOT(wires=[wires[i], wires[(i + 1) % len(wires)]])


def make_qnode(entangle: bool = True, n_layers: int = N_LAYERS,
               n_qubits: int = N_QUBITS, device_name: str = DEVICE):
    dev = qml.device(device_name, wires=n_qubits)

    @qml.qnode(dev, interface="torch", diff_method="backprop")
    def circuit(inputs, weights):
        qml.AngleEmbedding(inputs, wires=range(n_qubits), rotation="Y")
        for l in range(n_layers):
            variational_layer(weights[l], wires=range(n_qubits), entangle=entangle)
        return [qml.expval(qml.PauliZ(w)) for w in range(n_qubits)]

    weight_shapes = {"weights": (n_layers, n_qubits, 2)}
    return circuit, weight_shapes


# Quick sanity print of the circuit Control A will use
_circuit, _shapes = make_qnode(entangle=True)
print("Weight shape (n_layers, n_qubits, 2 rotations):", _shapes["weights"])
print(qml.draw(_circuit)(X_train_enc[0], np.zeros(_shapes["weights"])))


Weight shape (n_layers, n_qubits, 2 rotations): (2, 6, 2)
0: ─╭AngleEmbedding(M0)──RY(0.00)──RZ(0.00)─╭●─────────────╭X──RY(0.00)──RZ(0.00)─╭●──────────── ···
1: ─├AngleEmbedding(M0)──RY(0.00)──RZ(0.00)─╰X─╭●──────────│───RY(0.00)──RZ(0.00)─╰X─╭●───────── ···
2: ─├AngleEmbedding(M0)──RY(0.00)──RZ(0.00)────╰X─╭●───────│───RY(0.00)──RZ(0.00)────╰X─╭●────── ···
3: ─├AngleEmbedding(M0)──RY(0.00)──RZ(0.00)───────╰X─╭●────│───RY(0.00)──RZ(0.00)───────╰X─╭●─── ···
4: ─├AngleEmbedding(M0)──RY(0.00)──RZ(0.00)──────────╰X─╭●─│───RY(0.00)──RZ(0.00)──────────╰X─╭● ···
5: ─╰AngleEmbedding(M0)──RY(0.00)──RZ(0.00)─────────────╰X─╰●──RY(0.00)──RZ(0.00)─────────────╰X ···

0: ··· ─╭X─┤  <Z>
1: ··· ─│──┤  <Z>
2: ··· ─│──┤  <Z>
3: ··· ─│──┤  <Z>
4: ··· ─│──┤  <Z>
5: ··· ─╰●─┤  <Z>

M0 = 
[0.6843711  0.8056808  0.92189234 2.0750089  1.9608922  1.5162287 ]


## 3. Models

In [6]:

class HybridQNN(torch.nn.Module):
    """Classical pre-layer -> quantum layer -> classical output layer.

    Flags reproduce all four ablation controls from EXECUTION_PLAN.md with
    ONE class:
      Control A (full hybrid):        entangle=True,  quantum_trainable=True
      Control C (entanglement-ablated): entangle=False, quantum_trainable=True
      Control D (fixed/untrained-quantum): entangle=True, quantum_trainable=False
    """

    def __init__(self, n_features: int, n_qubits: int = N_QUBITS,
                 entangle: bool = True, quantum_trainable: bool = True):
        super().__init__()
        self.pre = torch.nn.Linear(n_features, n_qubits)
        circuit_fn, weight_shapes = make_qnode(entangle=entangle, n_qubits=n_qubits)
        self.q_layer = qml.qnn.TorchLayer(circuit_fn, weight_shapes)
        if not quantum_trainable:
            for p in self.q_layer.parameters():
                p.requires_grad = False
        self.post = torch.nn.Linear(n_qubits, 1)

    def forward(self, x):
        # keep pre-layer output within a sane angle range for encoding
        x = torch.tanh(self.pre(x)) * (torch.pi / 2)
        x = self.q_layer(x)
        x = self.post(x)
        return torch.sigmoid(x).squeeze(-1)


def count_trainable_params(model: torch.nn.Module) -> int:
    return sum(p.numel() for p in model.parameters() if p.requires_grad)


class ClassicalControl(torch.nn.Module):
    """Control B — same input/output shape, quantum layer replaced by an
    MLP hidden layer sized to match Control A's trainable parameter count."""

    def __init__(self, n_features: int, hidden_dim: int):
        super().__init__()
        self.net = torch.nn.Sequential(
            torch.nn.Linear(n_features, hidden_dim),
            torch.nn.Tanh(),
            torch.nn.Linear(hidden_dim, 1),
        )

    def forward(self, x):
        return torch.sigmoid(self.net(x)).squeeze(-1)


def build_param_matched_classical(n_features: int, target_params: int) -> ClassicalControl:
    """Search hidden_dim so ClassicalControl's param count is the closest
    match (from above) to target_params. Reported honestly either way — see
    EXECUTION_PLAN.md risk #3 on parameter-matching being subtle."""
    best_model, best_diff = None, None
    for hidden_dim in range(1, 64):
        m = ClassicalControl(n_features, hidden_dim)
        n_params = count_trainable_params(m)
        diff = abs(n_params - target_params)
        if best_diff is None or diff < best_diff:
            best_model, best_diff = m, diff
        if n_params >= target_params:
            break
    return best_model


## 4. Shared training & evaluation harness (used identically by every control)

In [7]:

def train_model(model: torch.nn.Module, X_train, y_train, epochs: int = 60,
                 lr: float = 0.05, seed: int = SEED) -> dict:
    set_seed(seed)
    opt = torch.optim.Adam(filter(lambda p: p.requires_grad, model.parameters()), lr=lr)
    loss_fn = torch.nn.BCELoss()

    history = []
    t0 = time.time()
    for epoch in range(epochs):
        opt.zero_grad()
        preds = model(X_train)
        loss = loss_fn(preds, y_train)
        loss.backward()
        opt.step()
        history.append(loss.item())
        if epoch % 10 == 0 or epoch == epochs - 1:
            print(f"  epoch {epoch:3d}  loss {loss.item():.4f}")
    train_time = time.time() - t0
    return {"history": history, "train_time_sec": train_time}


def evaluate_model(model: torch.nn.Module, X_test, y_test_np: np.ndarray) -> dict:
    model.eval()
    with torch.no_grad():
        probs = model(X_test).numpy()
    preds = (probs >= 0.5).astype(int)

    tn, fp, fn, tp = confusion_matrix(y_test_np, preds).ravel()
    specificity = tn / (tn + fp) if (tn + fp) > 0 else float("nan")

    return {
        "accuracy": accuracy_score(y_test_np, preds),
        "precision": precision_score(y_test_np, preds, zero_division=0),
        "recall_sensitivity": recall_score(y_test_np, preds, zero_division=0),
        "specificity": specificity,
        "f1": f1_score(y_test_np, preds, zero_division=0),
        "auc_roc": roc_auc_score(y_test_np, probs),
        "confusion_matrix": {"tn": int(tn), "fp": int(fp), "fn": int(fn), "tp": int(tp)},
    }


## 5. Run — Control A (full hybrid) vs. Control B (classical, param-matched)

Single seed, one honest number. This is the Phase 1 exit criterion — do not
move on to the multi-seed sweep (Phase 2) or the dashboard (Phase 4) until
this cell has produced a real result you'd defend to a judge.


In [8]:

n_features = X_train_t.shape[1]

print("=== Control A: Full Hybrid (entangled, trainable quantum layer) ===")
model_a = HybridQNN(n_features, entangle=True, quantum_trainable=True)
n_params_a = count_trainable_params(model_a)
print(f"Trainable params: {n_params_a}")
train_info_a = train_model(model_a, X_train_t, y_train_t, epochs=60)
metrics_a = evaluate_model(model_a, X_test_t, y_test)
metrics_a.update({"n_params": n_params_a, "train_time_sec": train_info_a["train_time_sec"]})
print(json.dumps(metrics_a, indent=2))


=== Control A: Full Hybrid (entangled, trainable quantum layer) ===
Trainable params: 73


  epoch   0  loss 0.7624


  epoch  10  loss 0.5276


  epoch  20  loss 0.2492


  epoch  30  loss 0.0967


  epoch  40  loss 0.0815


  epoch  50  loss 0.0761


  epoch  59  loss 0.0733
{
  "accuracy": 0.9824561403508771,
  "precision": 1.0,
  "recall_sensitivity": 0.9523809523809523,
  "specificity": 1.0,
  "f1": 0.975609756097561,
  "auc_roc": 0.9983465608465609,
  "confusion_matrix": {
    "tn": 72,
    "fp": 0,
    "fn": 2,
    "tp": 40
  },
  "n_params": 73,
  "train_time_sec": 3.476508378982544
}


In [9]:

print("=== Control B: Classical, parameter-matched ===")
model_b = build_param_matched_classical(n_features, target_params=n_params_a)
n_params_b = count_trainable_params(model_b)
print(f"Trainable params: {n_params_b} (target was {n_params_a})")
train_info_b = train_model(model_b, X_train_t, y_train_t, epochs=60)
metrics_b = evaluate_model(model_b, X_test_t, y_test)
metrics_b.update({"n_params": n_params_b, "train_time_sec": train_info_b["train_time_sec"]})
print(json.dumps(metrics_b, indent=2))


=== Control B: Classical, parameter-matched ===
Trainable params: 73 (target was 73)
  epoch   0  loss 0.6781
  epoch  10  loss 0.4103
  epoch  20  loss 0.1756
  epoch  30  loss 0.1043
  epoch  40  loss 0.0845
  epoch  50  loss 0.0771
  epoch  59  loss 0.0736
{
  "accuracy": 0.9736842105263158,
  "precision": 0.975609756097561,
  "recall_sensitivity": 0.9523809523809523,
  "specificity": 0.9861111111111112,
  "f1": 0.963855421686747,
  "auc_roc": 0.998015873015873,
  "confusion_matrix": {
    "tn": 71,
    "fp": 1,
    "fn": 2,
    "tp": 40
  },
  "n_params": 73,
  "train_time_sec": 0.036218881607055664
}


In [10]:

# Also log a standard, non-quantum baseline (Logistic Regression / RBF-SVM)
# for a sanity check against the wider ML literature on this dataset —
# distinct from Control B, which exists specifically to param-match the
# hybrid model for the ablation study.
print("=== Reference baseline: Logistic Regression (sanity check, not a control) ===")
logreg = LogisticRegression(max_iter=1000, random_state=SEED).fit(X_train_enc, y_train)
probs_lr = logreg.predict_proba(X_test_enc)[:, 1]
preds_lr = (probs_lr >= 0.5).astype(int)
print(f"  accuracy={accuracy_score(y_test, preds_lr):.3f}  auc={roc_auc_score(y_test, probs_lr):.3f}")


=== Reference baseline: Logistic Regression (sanity check, not a control) ===
  accuracy=0.965  auc=0.997


In [11]:

results_mvp = {
    "dataset": "WDBC",
    "seed": SEED,
    "n_qubits": N_QUBITS,
    "n_layers": N_LAYERS,
    "control_A_full_hybrid": metrics_a,
    "control_B_classical_matched": metrics_b,
}

out_path = RESULTS_DIR / "phase1_mvp_wdbc.json"
out_path.write_text(json.dumps(results_mvp, indent=2))
print(f"Saved: {out_path}")

pd.DataFrame([
    {"model": "A: Full Hybrid", **{k: v for k, v in metrics_a.items() if k != "confusion_matrix"}},
    {"model": "B: Classical (matched)", **{k: v for k, v in metrics_b.items() if k != "confusion_matrix"}},
])


Saved: results\phase1_mvp_wdbc.json


,model,accuracy,precision,recall_sensitivity,specificity,f1,auc_roc,n_params,train_time_sec
0,A: Full Hybrid,0.982456,1.00000,0.952381,1.000000,0.975610,0.998347,73,3.476508
1,B: Classical (matched),0.973684,0.97561,0.952381,0.986111,0.963855,0.998016,73,0.036219


## 6. QSVM fallback path (golden rule — use this if the VQC above won't train reliably)

Separate, independent path: quantum kernel via a fixed feature map, fed into
a classical SVM with a precomputed kernel. No gradient descent through a
quantum circuit, so it sidesteps VQC training-instability entirely. Kept
ready here but **not required** if Control A above already trained cleanly.


In [12]:

def train_eval_qsvm(X_train_enc, y_train, X_test_enc, y_test_np, n_qubits=N_QUBITS, reps=2):
    from qiskit_machine_learning.kernels import FidelityQuantumKernel
    # Qiskit >=2.1 deprecated the ZZFeatureMap *class* (removed in 3.0) in
    # favour of the zz_feature_map *function*; support both.
    try:
        from qiskit.circuit.library import zz_feature_map
        feature_map = zz_feature_map(feature_dimension=n_qubits, reps=reps)
    except ImportError:
        from qiskit.circuit.library import ZZFeatureMap
        feature_map = ZZFeatureMap(feature_dimension=n_qubits, reps=reps)

    qkernel = FidelityQuantumKernel(feature_map=feature_map)

    t0 = time.time()
    K_train = qkernel.evaluate(x_vec=X_train_enc)
    svm = SVC(kernel="precomputed", probability=True, random_state=SEED)
    svm.fit(K_train, y_train)
    train_time = time.time() - t0

    K_test = qkernel.evaluate(x_vec=X_test_enc, y_vec=X_train_enc)
    probs = svm.predict_proba(K_test)[:, 1]
    preds = (probs >= 0.5).astype(int)

    tn, fp, fn, tp = confusion_matrix(y_test_np, preds).ravel()
    specificity = tn / (tn + fp) if (tn + fp) > 0 else float("nan")
    return {
        "accuracy": accuracy_score(y_test_np, preds),
        "precision": precision_score(y_test_np, preds, zero_division=0),
        "recall_sensitivity": recall_score(y_test_np, preds, zero_division=0),
        "specificity": specificity,
        "f1": f1_score(y_test_np, preds, zero_division=0),
        "auc_roc": roc_auc_score(y_test_np, probs),
        "confusion_matrix": {"tn": int(tn), "fp": int(fp), "fn": int(fn), "tp": int(tp)},
        "train_time_sec": train_time,
    }

# Uncomment to actually run the fallback (kernel evaluation is O(n^2) circuit
# evaluations, so only run this on the full 455-sample train set if Control A
# above genuinely failed to converge — otherwise it just burns Kaggle time):
#
# qsvm_metrics = train_eval_qsvm(X_train_enc, y_train, X_test_enc, y_test)
# print(json.dumps(qsvm_metrics, indent=2))


## 7. What this notebook does NOT do yet (by design — see EXECUTION_PLAN.md)

- No multi-seed loop, no dataset-size sweep, no Wilcoxon/Holm-Bonferroni stats
  — that's Phase 2 (`02_ablation_sweep.ipynb`), which will import the
  `HybridQNN`, `train_model`, and `evaluate_model` functions defined above
  unchanged and just loop them over seeds/sizes/controls.
- No Controls C or D runs — the flags exist (`entangle`, `quantum_trainable`)
  but Phase 1's job is only to validate A vs. B.
- No Heart Disease / Parkinson's datasets — WDBC only, per the staged
  MUST-build order.
- No IBM hardware calls — simulator only. The hardware run reuses these exact
  trained weights for inference, later, on a separate small notebook.
- No claims. Only report `metrics_a` / `metrics_b` as computed above — never
  a target/expected number presented as an achieved result.


## 8. Clinical Threshold Tuning & ROC Operating Points

Evaluates the trained Control A hybrid model across operating thresholds
($\tau \in [0.05, 0.95]$). Identifies two key clinical decision points:
1. **Balanced triage point**: Minimizes $|\text{Sensitivity} - \text{Specificity}|$.
2. **High-sensitivity zero-miss protocol**: Lowest threshold ensuring $\ge 98\%$ sensitivity,
   quantifying the associated false-positive specificity cost.


In [13]:
import pickle
from sklearn.metrics import roc_curve, auc
import matplotlib.pyplot as plt

# Ensure model_a is in eval mode and get test probabilities
model_a.eval()
with torch.no_grad():
    test_probs = model_a(X_test_t).numpy()

y_test_np = y_test if isinstance(y_test, np.ndarray) else np.array(y_test)

# Threshold sweep from 0.05 to 0.95 (step 0.05)
thresholds = np.arange(0.05, 0.96, 0.05)
records = []
for th in thresholds:
    preds = (test_probs >= th).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_test_np, preds).ravel()
    sens = tp / (tp + fn) if (tp + fn) > 0 else 0.0
    spec = tn / (tn + fp) if (tn + fp) > 0 else 0.0
    acc = (tp + tn) / (tp + tn + fp + fn)
    records.append({
        "threshold": round(float(th), 2),
        "sensitivity": round(float(sens), 4),
        "specificity": round(float(spec), 4),
        "accuracy": round(float(acc), 4),
        "tn": int(tn), "fp": int(fp), "fn": int(fn), "tp": int(tp)
    })

tuning_df = pd.DataFrame(records)
print("=== WDBC Control A Clinical Threshold Tuning Sweep ===")
print(tuning_df[["threshold", "sensitivity", "specificity", "accuracy"]].to_string(index=False))

# Operating Point (a): Balanced (|sensitivity - specificity| minimized)
diff = np.abs(tuning_df["sensitivity"] - tuning_df["specificity"])
idx_balanced = int(np.argmin(diff))
balanced_pt = tuning_df.iloc[idx_balanced]

# Operating Point (b): High-Sensitivity (lowest threshold where sensitivity >= 0.98)
high_sens_candidates = tuning_df[tuning_df["sensitivity"] >= 0.98]
if len(high_sens_candidates) > 0:
    high_sens_pt = high_sens_candidates.iloc[0]
else:
    high_sens_pt = tuning_df.iloc[0]

spec_cost = 1.0 - high_sens_pt["specificity"]

print(f"\n[Operating Point 1: Balanced Diagnostic Triage]")
print(f"  Threshold:   tau = {balanced_pt['threshold']:.2f}")
print(f"  Sensitivity: {balanced_pt['sensitivity']*100:.2f}%")
print(f"  Specificity: {balanced_pt['specificity']*100:.2f}%")
print(f"  Accuracy:    {balanced_pt['accuracy']*100:.2f}%")

print(f"\n[Operating Point 2: High-Sensitivity Zero-Miss Protocol]")
print(f"  Threshold:   tau = {high_sens_pt['threshold']:.2f}")
print(f"  Sensitivity: {high_sens_pt['sensitivity']*100:.2f}% (>= 98% target satisfied)")
print(f"  Specificity: {high_sens_pt['specificity']*100:.2f}% (Specificity Cost: {spec_cost*100:.2f}% False Positives)")
print(f"  Accuracy:    {high_sens_pt['accuracy']*100:.2f}%")

# Save threshold tuning table
csv_out = RESULTS_DIR / "wdbc_threshold_tuning.csv"
tuning_df.to_csv(csv_out, index=False)
print(f"\nSaved threshold table: {csv_out}")

# ROC Curve Plot
fpr, tpr, roc_thresh = roc_curve(y_test_np, test_probs)
roc_auc = auc(fpr, tpr)

plt.figure(figsize=(7, 6), dpi=300)
plt.plot(fpr, tpr, color="#2b5c8f", lw=2.5, label=f"Control A Hybrid VQC (AUC = {roc_auc:.4f})")
plt.plot([0, 1], [0, 1], color="#999999", lw=1.2, linestyle="--", label="Chance Baseline")

# Mark Balanced operating point (FPR = 1 - specificity)
fpr_bal = 1.0 - balanced_pt["specificity"]
tpr_bal = balanced_pt["sensitivity"]
plt.scatter([fpr_bal], [tpr_bal], color="#e07a5f", s=130, zorder=5,
            edgecolor="black", lw=1.5,
            label=f"Balanced Point (tau={balanced_pt['threshold']:.2f}, Sens={tpr_bal:.2f}, Spec={balanced_pt['specificity']:.2f})")

# Mark High-Sensitivity operating point
fpr_high = 1.0 - high_sens_pt["specificity"]
tpr_high = high_sens_pt["sensitivity"]
plt.scatter([fpr_high], [tpr_high], color="#2a9d8f", s=130, zorder=5, marker="^",
            edgecolor="black", lw=1.5,
            label=f"High-Sens Point (tau={high_sens_pt['threshold']:.2f}, Sens={tpr_high:.2f}, Spec={high_sens_pt['specificity']:.2f})")

plt.xlabel("False Positive Rate (1 - Specificity)", fontsize=11, fontweight="bold")
plt.ylabel("True Positive Rate (Sensitivity / Recall)", fontsize=11, fontweight="bold")
plt.title("WDBC Control A: ROC Curve & Clinical Operating Points", fontsize=12, fontweight="bold", pad=12)
plt.grid(True, linestyle=":", alpha=0.6)
plt.legend(loc="lower right", fontsize=9, framealpha=0.95)
plt.tight_layout()

roc_out = RESULTS_DIR / "wdbc_roc_curve.png"
plt.savefig(roc_out)
plt.close()
print(f"Saved ROC curve plot: {roc_out}")

# Save model weights & fitted preprocessors for the Part 3 platform
import os
for pdir in [Path("platform/backend"), Path("../platform/backend")]:
    pdir.mkdir(parents=True, exist_ok=True)
    torch.save(model_a.state_dict(), pdir / "model_weights.pt")
    with open(pdir / "preprocessor.pkl", "wb") as f:
        pickle.dump({"scaler": scaler, "pca": pca, "angle_scaler": angle_scaler}, f)
print("Saved model weights & preprocessor to platform/backend")


=== WDBC Control A Clinical Threshold Tuning Sweep ===


 threshold  sensitivity  specificity  accuracy
      0.05       1.0000       0.9028    0.9386
      0.10       1.0000       0.9306    0.9561
      0.15       0.9762       0.9583    0.9649
      0.20       0.9762       0.9583    0.9649
      0.25       0.9762       0.9583    0.9649
      0.30       0.9762       0.9722    0.9737
      0.35       0.9762       0.9722    0.9737
      0.40       0.9762       0.9861    0.9825
      0.45       0.9762       1.0000    0.9912
      0.50       0.9524       1.0000    0.9825
      0.55       0.9524       1.0000    0.9825
      0.60       0.9286       1.0000    0.9737
      0.65       0.9048       1.0000    0.9649
      0.70       0.9048       1.0000    0.9649
      0.75       0.9048       1.0000    0.9649
      0.80       0.8810       1.0000    0.9561
      0.85       0.8810       1.0000    0.9561
      0.90       0.8333       1.0000    0.9386
      0.95       0.7619       1.0000    0.9123

[Operating Point 1: Balanced Diagnostic Triage]
  Threshol

Saved ROC curve plot: results\wdbc_roc_curve.png
Saved model weights & preprocessor to platform/backend
